In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score, roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
df = pd.read_csv("train(1).csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.columns

In [ ]:
df['gender'].value_counts()

In [ ]:
for col in df.columns:
    print(f"\nValue counts for column: {col}")
    print(df[col].value_counts(dropna=False))
    print("-" * 30)

In [ ]:
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100

print(pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_percent
}).sort_values(by="Missing %", ascending=False))

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
cat_cols = df.select_dtypes(include=['object']).columns

for col in cat_cols:
    print(f"\nColumn: {col}")
    print(df[col].value_counts())

In [ ]:
import matplotlib.pyplot as plt

num_cols = df.select_dtypes(include=['int64', 'float64']).columns

df[num_cols].hist(figsize=(15,10))
plt.tight_layout()
plt.show()

In [ ]:
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print(f"{col}: {len(outliers)} outliers")

In [ ]:
import seaborn as sns

plt.figure(figsize=(12,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.show()

In [ ]:
print(df['readmitted_30d'].value_counts())
print(df['readmitted_30d'].value_counts(normalize=True) * 100)

In [ ]:
# 11. FEATURE IMPORTANCE (Random Forest)
from sklearn.ensemble import RandomForestClassifier
import pandas as pd # Ensure pandas is imported
import matplotlib.pyplot as plt # Ensure matplotlib is imported

rf = RandomForestClassifier(class_weight='balanced', random_state=42)

num_features = X_train.shape[1]
feature_names = [f'feature_{i}' for i in range(num_features)]

rf.fit(X_train, y_train)

importance = pd.Series(rf.feature_importances_, index=feature_names)
importance = importance.sort_values(ascending=False)

plt.figure()
importance.head(18).plot(kind='barh')
plt.title("Top 10 Feature Importance")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
plt.figure(figsize=(10,6))

sns.kdeplot(
    data=df,
    x='glucose_level_mgdl',
    hue='gender',
    fill=True,
    common_norm=False,
    alpha=0.4
)

plt.title("Glucose Distribution by Gender")
plt.xlabel("Glucose Level (mg/dL)")
plt.ylabel("Density")
plt.show()

# 3. BOXPLOT
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x='gender',
    y='glucose_level_mgdl'
)

plt.title("Glucose Level by Gender")
plt.show()

In [ ]:
def split_data(df, target='readmitted_30d'):
    X = df.drop(columns=[target])
    y = df[target]

    return train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

class DataPreprocessor:
    def __init__(self):
        self.medians = {}
        self.scaler = StandardScaler()
        self.num_cols = None
        self.cat_cols = None

    def fit(self, df):
        df = df.copy()

        df['admission_date'] = pd.to_datetime(df['admission_date'], format='mixed')

        df['month'] = df['admission_date'].dt.month
        df['day'] = df['admission_date'].dt.day

        df.loc[df['age'] > 100, 'age'] = df['age'].median()

        self.medians['glucose'] = df['glucose_level_mgdl'].median()

        for col in ['length_of_stay_days', 'creatinine_mgdl']:
            df[col] = np.log1p(df[col])

        df['age_los'] = df['age'] * df['length_of_stay_days']
        df['comorbidity_los'] = df['charlson_comorbidity_index'] * df['length_of_stay_days']

        df.drop(columns=['patient_id', 'admission_date'], inplace=True)

        self.num_cols = df.select_dtypes(include=['int64', 'float64']).columns
        self.cat_cols = df.select_dtypes(include=['object']).columns

        self.scaler.fit(df[self.num_cols])

        return self

    def transform(self, df):
        df = df.copy()

        df = df.drop_duplicates()

        df['admission_date'] = pd.to_datetime(df['admission_date'], format='mixed')

        df['month'] = df['admission_date'].dt.month
        df['day'] = df['admission_date'].dt.day

        df.loc[df['age'] > 100, 'age'] = df['age'].median()

        df['glucose_level_mgdl'].fillna(self.medians['glucose'], inplace=True)

        for col in ['length_of_stay_days', 'creatinine_mgdl']:
            df[col] = np.log1p(df[col])

        df['age_los'] = df['age'] * df['length_of_stay_days']
        df['comorbidity_los'] = df['charlson_comorbidity_index'] * df['length_of_stay_days']

        df.drop(columns=['patient_id', 'admission_date'], inplace=True)

        df = pd.get_dummies(df, columns=self.cat_cols, drop_first=True)

        df = df.reindex(columns=self.num_cols.union(df.columns), fill_value=0)

        def cap_outliers(col):
            Q1 = col.quantile(0.25)
            Q3 = col.quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            return col.clip(lower, upper)

        df[self.num_cols] = df[self.num_cols].apply(cap_outliers)

        # Scaling
        df[self.num_cols] = self.scaler.transform(df[self.num_cols])

        return df

    def fit_transform(self, df):
        return self.fit(df).transform(df)


X_train, X_test, y_train, y_test = split_data(df)

preprocessor = DataPreprocessor()

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000),
    "Random Forest": RandomForestClassifier(class_weight='balanced'),
    "XGBoost": XGBClassifier(scale_pos_weight=10, eval_metric='logloss')
}

results = {}

for name, model in models.items():
    print(f"\n===== {name} =====")

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]

    print(classification_report(y_test, y_pred))

    auc = roc_auc_score(y_test, y_prob)
    print("ROC-AUC:", auc)

    results[name] = auc


plt.figure()
plt.bar(results.keys(), results.values())
plt.title("Model Comparison (ROC-AUC)")
plt.ylabel("AUC Score")
plt.xticks(rotation=20)
plt.show()


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Convert to tensors
X_train = torch.tensor(X_train.astype(np.float32).values, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.float32)

X_test = torch.tensor(X_test.astype(np.float32).values, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.float32)

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TabularDataset(X_train, y_train)
test_dataset = TabularDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
class TabularNN(nn.Module):
    def __init__(self, input_dim):
        super(TabularNN, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TabularNN(input_dim=X_train.shape[1]).to(device)

# Handle class imbalance
pos_weight = torch.tensor([len(y_train)/sum(y_train)]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 20
train_losses = []

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch).squeeze()
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)

    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

In [ ]:
plt.plot(train_losses)
plt.title("Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

In [ ]:
model.eval()

y_preds = []
y_probs = []

with torch.no_grad():
    for X_batch, _ in test_loader:
        X_batch = X_batch.to(device)

        outputs = model(X_batch).squeeze()
        probs = torch.sigmoid(outputs)

        y_probs.extend(probs.cpu().numpy())
        y_preds.extend((probs > 0.5).cpu().numpy())

y_preds = np.array(y_preds)
y_probs = np.array(y_probs)

In [ ]:
print(classification_report(y_test, y_preds))
print("ROC-AUC:", roc_auc_score(y_test, y_probs))

In [ ]:
threshold = 0.3  # increase recall

y_preds_custom = (y_probs > threshold).astype(int)

print("With Threshold =", threshold)
print(classification_report(y_test, y_preds_custom))

In [ ]:
for t in [0.3, 0.4, 0.5, 0.6, 0.7,0.8,0.7692214]:
    y_pred = (y_probs > t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, y_probs)

plt.plot(thresholds, precision[:-1], label='Precision')
plt.plot(thresholds, recall[:-1], label='Recall')
plt.legend()
plt.xlabel("Threshold")
plt.title("Precision vs Recall")
plt.show()

In [ ]:
f1_scores = 2 * (precision * recall) / (precision + recall)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("Best Threshold:", best_threshold)

In [ ]:
import joblib
import os
import torch

In [ ]:
import os
os.makedirs("models", exist_ok=True)

In [ ]:
import joblib

joblib.dump(model, "models/model.pkl")
joblib.dump(preprocessor, "models/preprocessor.pkl")

print("✅ Saved successfully")

In [ ]:
from google.colab import files

files.download("models/model.pkl")
files.download("models/preprocessor.pkl")

In [ ]:
torch.save(model.state_dict(), 'model.pth')

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

class HospitalFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        # 1. Date features
        X['admission_date'] = pd.to_datetime(X['admission_date'], format='mixed')
        X['month'] = X['admission_date'].dt.month
        X['day'] = X['admission_date'].dt.day

        # 2. Handle Age outliers (using 100 as threshold based on previous logic)
        X.loc[X['age'] > 100, 'age'] = X['age'].median()

        # 3. Log transforms for skewed features
        X['length_of_stay_days'] = np.log1p(X['length_of_stay_days'])
        X['creatinine_mgdl'] = np.log1p(X['creatinine_mgdl'])

        # 4. Interaction terms
        X['age_los'] = X['age'] * X['length_of_stay_days']
        X['comorbidity_los'] = X['charlson_comorbidity_index'] * X['length_of_stay_days']

        # 5. Drop ID and raw date
        cols_to_drop = ['patient_id', 'admission_date']
        X = X.drop(columns=[c for c in cols_to_drop if c in X.columns])
        return X

# Group columns for processing
num_features = ['age', 'admission_type', 'discharge_destination', 'length_of_stay_days',
                'charlson_comorbidity_index', 'prior_admissions_1yr', 'n_medications_discharge',
                'glucose_level_mgdl', 'blood_pressure_systolic', 'sodium_meql',
                'creatinine_mgdl', 'haemoglobin_gdl', 'month', 'day', 'age_los', 'comorbidity_los']
cat_features = ['gender', 'discharge_day_of_week', 'insurance_type']

# Define the full pipeline
preprocessing_pipeline = Pipeline(steps=[
    ('feature_engineer', HospitalFeatureEngineer()),
    ('transformer', ColumnTransformer(transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_features),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_features)
    ]))
])

# Re-split and process raw data
X = df.drop(columns=['readmitted_30d'])
y = df['readmitted_30d']
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Apply the pipeline
X_train_final = preprocessing_pipeline.fit_transform(X_train_raw)
X_test_final = preprocessing_pipeline.transform(X_test_raw)

print(f"Processed Training Shape: {X_train_final.shape}")
print("Pipeline successfully created and fitted.")

In [ ]:
import joblib
import os

# Ensure the models directory exists
os.makedirs('models', exist_ok=True)

# Save the preprocessing pipeline
joblib.dump(preprocessing_pipeline, 'models/preprocessing_pipeline.pkl')

print("✅ Preprocessing pipeline saved to models/preprocessing_pipeline.pkl")